Getting the Data

In [2]:
import duckdb
import pandas as pd
db_path = "/Users/teddrako/Documents/GitHub/multi-factor-models/crsp/crsp_dataset.duckdb"
parquet_file = "/Users/teddrako/Documents/GitHub/multi-factor-models/crsp/CRSPdF20022025.parquet"

conn = duckdb.connect(db_path)

conn.execute(f"""
CREATE TABLE IF NOT EXISTS crsp_table AS
SELECT * FROM read_parquet('{parquet_file}')
""")

crsp_table_df = conn.execute("SELECT * FROM crsp_table").fetchdf()
crsp_table_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40802450,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000
40802451,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999
40802452,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000
40802453,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500


Cleaning Where Return is Zero

In [3]:
clean_crsp_df = conn.execute("""
SELECT *   
FROM crsp_table_df
WHERE ret IS NOT NULL
AND ret <> 0
""").fetchdf()

clean_crsp_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38829126,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000
38829127,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999
38829128,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000
38829129,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500


Convert Cusip 8 to 9 Digit

In [4]:
def cusip_convert(cusip8):
    # compute CUSIP check digit for an 8-character input
    values = {ch: i for i, ch in enumerate('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ*@#', start=0)}
    if cusip8 is None:
        return None
    s = str(cusip8).strip().upper()
    if len(s) < 8:
        return None
    s = s[:8]
    total = 0
    for i, ch in enumerate(s):
        v = values.get(ch, 0)
        if (i + 1) % 2 == 0:
            v *= 2
        total += v // 10 + v % 10
    return s + str((10 - (total % 10)) % 10)

# Fetch cleaned CRSP table into pandas, compute cusip9 in Python, and register with DuckDB
crsp_df = conn.execute("SELECT * FROM clean_crsp_df").fetchdf()
crsp_df['cusip9'] = crsp_df['cusip'].astype(str).str[:8].apply(cusip_convert)

# register the DataFrame as a DuckDB table for downstream SQL usage
conn.register('cusips9_df', crsp_df)

cusips9_df = crsp_df
cusips9_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,cusip9
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000,367204104
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000,05978R107
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000,233269109
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970,449507102
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000,001031103
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38829126,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000,92835K103
38829127,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999,G3323L100
38829128,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000,12503M108
38829129,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500,785135104


(can add dropping penny stocks and winsorizing daily returns)

Add Daily Market Cap (anchor factor)

In [5]:
market_cap_df = conn.execute("""
    SELECT *, abs(prc) * shrout * 1000 AS market_cap
    FROM cusips9_df
    """).fetchdf()

market_cap_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,cusip9,market_cap
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.10,-0.030568,-0.030568,6603.0,11.35000,11.49000,11.19,11.10000,367204104,2.848260e+07
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.60,-0.029323,-0.029323,55000.0,9.89000,9.60000,9.27,9.60000,05978R107,8.188800e+07
2,23326910,2002-01-02,10012,7969,1.0,1.0,20924.0,3.25,0.041667,0.041667,50975.0,3.25000,3.29000,3.24,3.06000,233269109,6.800300e+07
3,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.05,-0.160000,-0.160000,53250.0,1.05000,1.30000,1.03,1.04970,449507102,8.696100e+06
4,00103110,2002-01-02,10025,7975,1.0,1.0,7794.0,23.80,-0.006678,-0.006678,7030.0,23.90000,24.00000,23.80,23.40000,001031103,1.854972e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38829126,92835K10,2023-12-29,93426,53443,1.0,1.0,12510.0,34.07,0.013385,0.013385,94057.0,34.10000,34.50000,34.07,33.79000,92835K103,4.262157e+08
38829127,G3323L10,2023-12-29,93427,53445,1.0,1.0,36331.0,190.33,-0.033416,-0.033416,296974.0,190.32001,196.05000,190.12,189.50999,G3323L100,6.914879e+09
38829128,12503M10,2023-12-29,93429,53447,1.0,1.0,105556.0,178.56,0.004049,0.004049,521913.0,178.56000,179.09000,178.55,177.44000,12503M108,1.884808e+10
38829129,78513510,2023-12-29,93434,53427,1.0,1.0,43039.0,0.70,0.166667,0.166667,169185.0,0.70010,0.71495,0.70,0.58500,785135104,3.012730e+07


Value-Weighted Market Return

In [6]:
vw_market_return_df = conn.execute("""
    WITH lagged AS (
  SELECT
    date,
    permno,
    ret,
    LAG(market_cap) OVER (
      PARTITION BY permno
      ORDER BY date
    ) AS lag_mktcap
  FROM market_cap_df
  WHERE ret IS NOT NULL
    AND market_cap IS NOT NULL
),
weighted AS (
  SELECT
    date,
    permno,
    ret,
    lag_mktcap,
    SUM(lag_mktcap) OVER (PARTITION BY date) AS total_lag_mktcap
  FROM lagged
  WHERE lag_mktcap IS NOT NULL
    AND lag_mktcap > 0
)
SELECT
  date,
  SUM( (lag_mktcap / NULLIF(total_lag_mktcap, 0.0)) * ret ) AS market_factor_return,
  COUNT(*) AS n_stocks
FROM weighted
GROUP BY date
ORDER BY date
    """).fetchdf()

vw_market_return_df

,date,market_factor_return,n_stocks
0,2002-01-03,0.010148,6542
1,2002-01-04,0.006835,6836
2,2002-01-07,-0.006808,6900
3,2002-01-08,-0.002744,6845
4,2002-01-09,-0.004530,6885
...,...,...,...
5531,2023-12-22,0.002357,9135
5532,2023-12-26,0.005264,9170
5533,2023-12-27,0.002044,9117
5534,2023-12-28,-0.000062,9116


Ticker Upload

In [7]:
# read the ticker CSV into a DataFrame from ticker_data.csv
ticker_convert_df = pd.read_csv("/Users/teddrako/Documents/GitHub/multi-factor-models/ted_workspace/Research/ticker_data.csv")

ticker_convert_df

,Row ID,Ticker,BB Price Source,Nonsense,Cntry Terrtry (old),Country,Ind Sector (old),Sector,Ind Group (old),Group,CUSIP
0,1,VTNRQ,US,d,US,US,Energy,Energy,Oil&Gas,Oil&Gas,92534K107
1,2,1891569D,GR,d,US,US,Financial,Financial,Banks,Banks,858907108
2,3,2535201.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ignore
3,4,QUMU,US,d,US,US,Technology,Technology,Software,Software,749063103
4,5,40251W408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ignore
...,...,...,...,...,...,...,...,...,...,...,...
20548,20550,SCAP,US,d,US,US,Funds,Funds,Equity Fund,Equity Fund,81752T445
20549,20551,XIDE,US,d,US,US,Funds,Funds,Equity Fund,Equity Fund,33740F326
20550,20552,PRMN,US,d,NaN,NaN,Funds,Funds,Equity Fund,Equity Fund,66538R516
20551,20553,Q9379E105,NaN,NaN,US,US,NaN,NaN,NaN,NaN,Ignore


Ticker Clean

In [8]:
ticker_clean_df = conn.execute("""
    SELECT * EXCLUDE ('Row ID')
    FROM ticker_convert_df
    WHERE CUSIP != 'Ignore'
    """).fetchdf()

ticker_clean_df

,Ticker,BB Price Source,Nonsense,Cntry Terrtry (old),Country,Ind Sector (old),Sector,Ind Group (old),Group,CUSIP
0,VTNRQ,US,d,US,US,Energy,Energy,Oil&Gas,Oil&Gas,92534K107
1,1891569D,GR,d,US,US,Financial,Financial,Banks,Banks,858907108
2,QUMU,US,d,US,US,Technology,Technology,Software,Software,749063103
3,IEC,US,hd,US,US,Industrial,Industrial,Electronics,Electronics,44949L105
4,940991Q,US,d,US,US,Financial,Financial,REITS,REITS,144418100
...,...,...,...,...,...,...,...,...,...,...
15903,QOWZ,US,d,None,None,Funds,Funds,Equity Fund,Equity Fund,46138G466
15904,SCAP,US,d,US,US,Funds,Funds,Equity Fund,Equity Fund,81752T445
15905,XIDE,US,d,US,US,Funds,Funds,Equity Fund,Equity Fund,33740F326
15906,PRMN,US,d,None,None,Funds,Funds,Equity Fund,Equity Fund,66538R516


Country Loading

In [9]:
country_lookup_df = conn.execute("""
    SELECT
        m.*,
        t.Country AS Country
    FROM market_cap_df m
    LEFT JOIN ticker_clean_df t
        ON m.cusip9 = t.CUSIP
    WHERE t.country IS NOT NULL
""").fetchdf()

country_lookup_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,cusip9,market_cap,Country
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.1000,-0.030568,-0.030568,6603.0,11.35,11.490,11.19,11.1000,367204104,2.848260e+07,US
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.6000,-0.029323,-0.029323,55000.0,9.89,9.600,9.27,9.6000,05978R107,8.188800e+07,US
2,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.0500,-0.160000,-0.160000,53250.0,1.05,1.300,1.03,1.0497,449507102,8.696100e+06,US
3,46603210,2002-01-02,10026,7976,2.0,2.0,8654.0,24.8200,0.015133,0.015133,10544.0,24.84,24.830,24.37,23.4600,466032109,2.147923e+08,US
4,72913210,2002-01-02,10032,7980,1.0,1.0,41812.0,27.6200,0.039910,0.039910,521574.0,27.62,27.690,27.61,26.2100,729132100,1.154847e+09,US
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17523916,46428753,2023-12-27,89062,37493,1.0,1.0,800.0,65.0991,0.003807,0.003807,11896.0,65.21,65.330,65.17,65.0000,464287531,5.207928e+07,US
17523917,89346D10,2023-12-27,89068,41867,1.0,1.0,309865.0,8.2900,-0.008373,-0.008373,299166.0,8.30,8.375,8.29,8.2400,89346D107,2.568781e+09,CA
17523918,13646K10,2023-12-27,89125,41990,1.0,1.0,931791.0,79.7400,0.003524,0.003524,1460579.0,79.75,80.100,79.74,79.4200,13646K108,7.430101e+10,CA
17523919,46428746,2023-12-27,89129,37493,1.0,1.0,671400.0,75.4700,0.005328,0.005328,13103797.0,75.48,75.530,75.47,75.1100,464287465,5.067056e+10,US


Raw Country Factor Returns

In [10]:
country_factors_raw_df = conn.execute("""
WITH lagged AS (
    SELECT
        date,
        permno,
        Country,
        ret,
        LAG(market_cap) OVER (
            PARTITION BY permno
            ORDER BY date
        ) AS lag_mktcap
    FROM country_lookup_df
    WHERE ret IS NOT NULL
      AND market_cap IS NOT NULL
      AND Country IS NOT NULL
),

country_weights AS (
    SELECT
        date,
        Country,
        permno,
        ret,
        lag_mktcap,
        SUM(lag_mktcap) OVER (
            PARTITION BY date, Country
        ) AS country_total_lag_mktcap
    FROM lagged
    WHERE lag_mktcap IS NOT NULL
      AND lag_mktcap > 0
),

country_factor_raw AS (
    SELECT
        date,
        Country,
        SUM(
            (lag_mktcap / NULLIF(country_total_lag_mktcap, 0.0)) * ret
        ) AS country_ret_raw,
        COUNT(*) AS n_stocks
    FROM country_weights
    GROUP BY date, Country
)

SELECT
    date,
    Country,
    country_ret_raw,
    n_stocks
FROM country_factor_raw
ORDER BY date, Country;
""").fetchdf()

country_factors_raw_df

,date,Country,country_ret_raw,n_stocks
0,2002-01-03,AR,-0.010279,8
1,2002-01-03,AU,0.007747,4
2,2002-01-03,BD,0.002987,1
3,2002-01-03,BE,0.018975,1
4,2002-01-03,BZ,0.007187,2
...,...,...,...,...
242416,2023-12-29,U2,0.008664,2
242417,2023-12-29,UA,0.012013,2
242418,2023-12-29,UR,-0.006361,1
242419,2023-12-29,US,-0.004575,3962


Pure Country Risk (remove double counted market beta)

In [11]:
country_factors_pure_df = conn.execute("""
WITH joined AS (
    SELECT
        c.date,
        c.Country,
        c.country_ret_raw,
        c.n_stocks AS n_stocks_country,
        m.market_factor_return
    FROM country_factors_raw_df c
    JOIN vw_market_return_df m
      ON c.date = m.date
    WHERE c.country_ret_raw IS NOT NULL
      AND m.market_factor_return IS NOT NULL
),

means AS (
    SELECT
        Country,
        AVG(country_ret_raw)        AS mean_cty,
        AVG(market_factor_return)   AS mean_mkt
    FROM joined
    GROUP BY Country
),

beta AS (
    SELECT
        j.Country,
        SUM((j.country_ret_raw - mn.mean_cty) * (j.market_factor_return - mn.mean_mkt))
          / NULLIF(SUM((j.market_factor_return - mn.mean_mkt) * (j.market_factor_return - mn.mean_mkt)), 0.0)
          AS beta_c_m
    FROM joined j
    JOIN means mn
      ON j.Country = mn.Country
    GROUP BY j.Country
)

SELECT
    j.date,
    j.Country,
    j.country_ret_raw,
    j.market_factor_return,
    b.beta_c_m,
    (j.country_ret_raw - b.beta_c_m * j.market_factor_return) AS country_ret_orth,
    j.n_stocks_country
FROM joined j
JOIN beta b
  ON j.Country = b.Country
ORDER BY j.date, j.Country;
""").fetchdf()

country_factors_pure_df

,date,Country,country_ret_raw,market_factor_return,beta_c_m,country_ret_orth,n_stocks_country
0,2002-01-03,AR,-0.010279,0.010148,0.733079,-0.017718,8
1,2002-01-03,AU,0.007747,0.010148,1.270253,-0.005144,4
2,2002-01-03,BD,0.002987,0.010148,0.726808,-0.004389,1
3,2002-01-03,BE,0.018975,0.010148,0.813125,0.010723,1
4,2002-01-03,BZ,0.007187,0.010148,1.393004,-0.006950,2
...,...,...,...,...,...,...,...
242416,2023-12-29,U2,0.008664,-0.004036,0.328487,0.009990,2
242417,2023-12-29,UA,0.012013,-0.004036,1.090662,0.016416,2
242418,2023-12-29,UR,-0.006361,-0.004036,1.595706,0.000080,1
242419,2023-12-29,US,-0.004575,-0.004036,0.988708,-0.000584,3962


Industry Loading

In [12]:
industry_lookup_df = conn.execute("""
    SELECT
        m.*,
        t.Sector AS Industry
    FROM market_cap_df m
    LEFT JOIN ticker_clean_df t
        ON m.cusip9 = t.CUSIP
    WHERE t.Sector IS NOT NULL
""").fetchdf()

industry_lookup_df

,cusip,date,permno,permco,cfacpr,cfacshr,shrout,prc,ret,retx,vol,ask,askhi,bid,bidlo,cusip9,market_cap,Industry
0,36720410,2002-01-02,10001,7953,1.5,1.5,2566.0,11.100,-0.030568,-0.030568,6603.0,11.35,11.49,11.190,11.1000,367204104,28482600.0,Utilities
1,05978R10,2002-01-02,10002,7954,1.0,1.0,8530.0,9.600,-0.029323,-0.029323,55000.0,9.89,9.60,9.270,9.6000,05978R107,81888000.0,Financial
2,44950710,2002-01-02,10019,7971,1.0,1.0,8282.0,1.050,-0.160000,-0.160000,53250.0,1.05,1.30,1.030,1.0497,449507102,8696100.0,Industrial
3,46603210,2002-01-02,10026,7976,2.0,2.0,8654.0,24.820,0.015133,0.015133,10544.0,24.84,24.83,24.370,23.4600,466032109,214792280.0,"Consumer, Non-cyclical"
4,29402E10,2002-01-02,10028,7978,1.0,1.0,4913.0,3.350,-0.014706,-0.014706,1000.0,3.60,3.65,3.350,3.3500,29402E102,16458550.0,"Consumer, Cyclical"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29628921,74732410,2023-11-24,22229,58881,1.0,1.0,44323.0,1.550,0.107143,0.107143,129767.0,1.56,1.56,1.550,1.3500,747324101,68700650.0,"Consumer, Non-cyclical"
29628922,46138G53,2023-11-24,22230,52902,1.0,1.0,175.0,19.150,0.003932,0.003932,253.0,19.14,19.15,19.110,19.1200,46138G532,3351250.0,Funds
29628923,46138G54,2023-11-24,22231,52902,1.0,1.0,900.0,26.580,-0.001503,-0.001503,1719.0,26.58,26.58,26.560,26.5350,46138G540,23922000.0,Funds
29628924,76010Y10,2023-11-24,22232,58882,1.0,1.0,66552.0,0.599,0.071748,0.071748,226626.0,0.60,0.60,0.585,0.5408,76010Y103,39864648.0,"Consumer, Cyclical"


Raw Industry Factor Returns

In [13]:
industry_factors_raw_df = conn.execute("""
WITH lagged AS (
    SELECT
        date,
        permno,
        Industry,
        ret,
        LAG(market_cap) OVER (
            PARTITION BY permno
            ORDER BY date
        ) AS lag_mktcap
    FROM industry_lookup_df
    WHERE ret IS NOT NULL
      AND market_cap IS NOT NULL
      AND Industry IS NOT NULL
),

industry_weights AS (
    SELECT
        date,
        Industry,
        permno,
        ret,
        lag_mktcap,
        SUM(lag_mktcap) OVER (
            PARTITION BY date, Industry
        ) AS industry_total_lag_mktcap
    FROM lagged
    WHERE lag_mktcap IS NOT NULL
      AND lag_mktcap > 0
),

industry_factor_raw AS (
    SELECT
        date,
        Industry,
        SUM(
            (lag_mktcap / NULLIF(industry_total_lag_mktcap, 0.0)) * ret
        ) AS industry_ret_raw,
        COUNT(*) AS n_stocks
    FROM industry_weights
    GROUP BY date, Industry
)

SELECT
    date,
    Industry,
    industry_ret_raw,
    n_stocks
FROM industry_factor_raw
ORDER BY date, Industry;
""").fetchdf()

industry_factors_raw_df


,date,Industry,industry_ret_raw,n_stocks
0,2002-01-03,Basic Materials,0.015806,126
1,2002-01-03,Communications,0.023217,425
2,2002-01-03,"Consumer, Cyclical",0.005150,529
3,2002-01-03,"Consumer, Non-cyclical",-0.006609,815
4,2002-01-03,Diversified,0.002857,1
...,...,...,...,...
64073,2023-12-29,Funds,-0.003403,2817
64074,2023-12-29,Government,0.002893,1
64075,2023-12-29,Industrial,-0.002801,382
64076,2023-12-29,Technology,-0.003653,365


Pure Industry Risk

In [14]:
# Compute Pure Industry Risk: remove market and country effects
import numpy as np
import pandas as pd

# Verify required DataFrames exist
for name in ('industry_factors_raw_df','vw_market_return_df','country_factors_pure_df'):
    if name not in globals():
        raise RuntimeError(f"Required DataFrame '{name}' not found in notebook namespace")

# Pivot country pure returns into wide format (date x Country)
country_pivot = (
    country_factors_pure_df[['date', 'Country', 'country_ret_orth']]
    .pivot(index='date', columns='Country', values='country_ret_orth')
    .reset_index()
)

# Merge industry raws with market and country series
merged = (
    industry_factors_raw_df
    .merge(vw_market_return_df[['date','market_factor_return']], on='date', how='left')
    .merge(country_pivot, on='date', how='left')
)

# Prepare lists
country_cols = [c for c in country_pivot.columns if c != 'date']
industries = merged['Industry'].dropna().unique()

pure_rows = []
beta_rows = []

for ind in industries:
    sub = merged[merged['Industry'] == ind].reset_index(drop=True)
    R = sub['industry_ret_raw'].to_numpy()
    M = sub['market_factor_return'].to_numpy()

    # compute beta to market (no intercept) using rows where both values exist
    m_mask = ~np.isnan(R) & ~np.isnan(M)
    if m_mask.sum() >= 2 and np.nanstd(M[m_mask]) > 0:
        beta_m = np.linalg.lstsq(M[m_mask].reshape(-1,1), R[m_mask], rcond=None)[0][0]
    else:
        beta_m = 0.0

    # residual after removing market exposure (only where M available)
    residual = R.copy()
    valid_m = ~np.isnan(M)
    residual[valid_m] = R[valid_m] - beta_m * M[valid_m]

    # compute beta to each country separately (regress residual on country factor)
    beta_by_country = {}
    for c in country_cols:
        C = sub[c].to_numpy()
        c_mask = ~np.isnan(residual) & ~np.isnan(C)
        if c_mask.sum() >= 2 and np.nanstd(C[c_mask]) > 0:
            beta_sc = np.linalg.lstsq(C[c_mask].reshape(-1,1), residual[c_mask], rcond=None)[0][0]
        else:
            beta_sc = 0.0
        beta_by_country[c] = float(beta_sc)
        beta_rows.append({'Industry': ind, 'Country': c, 'beta': float(beta_sc)})

    # compute total adjustment and pure industry return
    M_f = np.nan_to_num(M, nan=0.0)
    adj = beta_m * M_f
    for c in country_cols:
        C = sub[c].to_numpy()
        adj += beta_by_country[c] * np.nan_to_num(C, nan=0.0)

    pure = R - adj

    # collect rows
    for i in range(len(sub)):
        pure_rows.append({
            'date': sub.loc[i,'date'],
            'Industry': ind,
            'industry_ret_raw': sub.loc[i,'industry_ret_raw'],
            'market_factor_return': sub.loc[i,'market_factor_return'],
            'industry_ret_pure': float(pure[i]) if not np.isnan(pure[i]) else None,
            'beta_market': float(beta_m)
        })

# Build DataFrames
industry_factors_pure_df = pd.DataFrame(pure_rows)
industry_betas_df = pd.DataFrame(beta_rows)

# Register with DuckDB for downstream SQL queries
conn.register('industry_factors_pure_df', industry_factors_pure_df)
conn.register('industry_betas_df', industry_betas_df)

print('Computed pure industry factors for', industry_factors_pure_df['Industry'].nunique(), 'industries')
industry_factors_pure_df

Computed pure industry factors for 12 industries


,date,Industry,industry_ret_raw,market_factor_return,industry_ret_pure,beta_market
0,2002-01-03,Basic Materials,0.015806,0.010148,0.008097,1.143628
1,2002-01-04,Basic Materials,0.020259,0.006835,0.014457,1.143628
2,2002-01-07,Basic Materials,-0.006428,-0.006808,-0.014049,1.143628
3,2002-01-08,Basic Materials,-0.013478,-0.002744,0.006458,1.143628
4,2002-01-09,Basic Materials,-0.001932,-0.004530,0.013394,1.143628
...,...,...,...,...,...,...
64073,2023-12-22,Utilities,0.003563,0.002357,0.002010,0.691132
64074,2023-12-26,Utilities,0.007128,0.005264,0.000567,0.691132
64075,2023-12-27,Utilities,-0.001092,0.002044,-0.003387,0.691132
64076,2023-12-28,Utilities,0.006667,-0.000062,0.010401,0.691132


Factor Loadings By Equity

In [15]:
# Estimate factor loadings per equity (by CUSIP)
import numpy as np
import pandas as pd
import re
from pathlib import Path

# helper to sanitize column names
def sanitize(name):
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(name)).strip("_")

# choose cusip column
cusip_col = 'cusip9' if 'cusip9' in market_cap_df.columns else ('cusip' if 'cusip' in market_cap_df.columns else None)
if cusip_col is None:
    raise RuntimeError('No CUSIP-like column found in market_cap_df')

# build factor matrix
factors = vw_market_return_df[['date','market_factor_return']].drop_duplicates()

# country factors (use orthogonalized country returns)
if 'country_factors_pure_df' in globals():
    country_pivot = (
        country_factors_pure_df[['date','Country','country_ret_orth']]
        .pivot(index='date', columns='Country', values='country_ret_orth')
        .reset_index()
    )
    # rename country columns
    country_cols = []
    for c in country_pivot.columns:
        if c == 'date':
            continue
        newc = 'cty_' + sanitize(c)
        country_cols.append(newc)
        country_pivot = country_pivot.rename(columns={c: newc})
    factors = factors.merge(country_pivot, on='date', how='left')
else:
    country_cols = []

# industry factors (pure)
if 'industry_factors_pure_df' in globals():
    industry_pivot = (
        industry_factors_pure_df[['date','Industry','industry_ret_pure']]
        .pivot(index='date', columns='Industry', values='industry_ret_pure')
        .reset_index()
    )
    industry_cols = []
    for s in industry_pivot.columns:
        if s == 'date':
            continue
        newc = 'ind_' + sanitize(s)
        industry_cols.append(newc)
        industry_pivot = industry_pivot.rename(columns={s: newc})
    factors = factors.merge(industry_pivot, on='date', how='left')
else:
    industry_cols = []

# full factor column order
factor_cols = ['market_factor_return'] + country_cols + industry_cols

print('Factor columns (total =', len(factor_cols), '):', factor_cols[:10], '...')

# merge stock returns with factors
stocks = market_cap_df.merge(factors, on='date', how='left')

# ensure ret column exists
if 'ret' not in stocks.columns:
    raise RuntimeError("market_cap_df must contain 'ret' column with returns")

# group by cusip and run regressions
min_obs = max(30, len(factor_cols) + 5)

wide_rows = []
long_rows = []
count = 0
for cusip, g in stocks.groupby(cusip_col):
    g = g.sort_values('date')
    y = g['ret'].to_numpy()
    # build X with factors, fill factor NaNs with 0 (consistent with previous steps)
    X = np.column_stack([g[col].to_numpy() for col in factor_cols]) if factor_cols else np.empty((len(g),0))
    X = np.nan_to_num(X, nan=0.0)
    valid = ~np.isnan(y)
    n_obs = valid.sum()
    if n_obs < min_obs:
        continue
    yv = y[valid]
    Xv = X[valid]
    # add intercept
    X_design = np.column_stack([np.ones(len(Xv)), Xv])
    try:
        coef, *_ = np.linalg.lstsq(X_design, yv, rcond=None)
    except Exception:
        continue
    alpha = float(coef[0])
    betas = coef[1:]
    # wide row
    row = {cusip_col: cusip, 'n_obs': int(n_obs), 'alpha': alpha}
    # fill factor betas
    for name, b in zip(factor_cols, betas):
        row['beta_' + name] = float(b)
        long_rows.append({cusip_col: cusip, 'factor': name, 'beta': float(b)})
    wide_rows.append(row)
    count += 1

print('Estimated loadings for', count, 'CUSIPs')

# produce DataFrames
loadings_wide_df = pd.DataFrame(wide_rows)
loadings_long_df = pd.DataFrame(long_rows)

# register and save
conn.register('loadings_wide_df', loadings_wide_df)
conn.register('loadings_long_df', loadings_long_df)

# write outputs (save to the Research/outputs folder)
# This ensures outputs are colocated with the notebook: ted_workspace/Research/outputs
out_dir = Path.cwd() / 'ted_workspace' / 'Research' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
loadings_wide_df.to_csv(out_dir / 'cusip_loadings_wide.csv', index=False)
loadings_long_df.to_csv(out_dir / 'cusip_loadings_long.csv', index=False)

print('Saved wide and long loadings to', str(out_dir), '(rows)', len(loadings_wide_df), len(loadings_long_df))
loadings_wide_df

Factor columns (total = 67 ): ['market_factor_return', 'cty_AR', 'cty_AU', 'cty_BD', 'cty_BE', 'cty_BZ', 'cty_CA', 'cty_CC', 'cty_CH', 'cty_CI'] ...
Estimated loadings for 19959 CUSIPs
Saved wide and long loadings to /Users/teddrako/Documents/GitHub/multi-factor-models/ted_workspace/Research/ted_workspace/Research/outputs (rows) 19959 1337253


,cusip9,n_obs,alpha,beta_market_factor_return,beta_cty_AR,beta_cty_AU,beta_cty_BD,beta_cty_BE,beta_cty_BZ,beta_cty_CA,...,beta_ind_Consumer_Cyclical,beta_ind_Consumer_Non_cyclical,beta_ind_Diversified,beta_ind_Energy,beta_ind_Financial,beta_ind_Funds,beta_ind_Government,beta_ind_Industrial,beta_ind_Technology,beta_ind_Utilities
0,000307108,1251,-0.001609,0.975736,0.126150,0.057386,-8.437695e-14,-0.052679,-0.084945,-0.940445,...,-0.327868,0.277601,-0.022685,0.068375,-0.771996,-2.671759,0.110655,0.221339,-0.847782,-0.350057
1,00032Q104,1338,-0.000463,0.793721,0.007701,0.295796,-2.436940e-14,0.044163,0.191688,0.571437,...,0.041280,0.891037,-0.020420,0.345415,0.633918,-1.992948,0.413623,-0.801888,-0.620049,-0.226723
2,000360206,5468,0.000529,1.118369,-0.010468,0.045201,-5.988268e-03,0.001915,-0.000182,-0.198529,...,0.091248,0.024453,-0.005766,-0.082550,-0.066306,-0.027480,-0.001746,0.613039,-0.127176,-0.094830
3,000361105,5481,0.000223,1.425338,0.057237,-0.009537,6.099019e-02,0.098455,0.017490,0.036780,...,0.465824,0.305997,0.040661,0.427869,0.435034,0.570607,-0.006007,1.320757,0.328607,0.075798
4,000375204,5272,0.000165,1.241287,0.003964,0.026373,4.576894e-02,0.061782,-0.002554,0.192800,...,0.042191,-0.201559,0.021117,-0.112964,0.147023,-0.037118,-0.000816,0.363655,0.050011,0.020682


In [21]:
loadings_wide_df

,cusip9,n_obs,alpha,beta_market_factor_return,beta_cty_AR,beta_cty_AU,beta_cty_BD,beta_cty_BE,beta_cty_BZ,beta_cty_CA,...,beta_ind_Consumer_Cyclical,beta_ind_Consumer_Non_cyclical,beta_ind_Diversified,beta_ind_Energy,beta_ind_Financial,beta_ind_Funds,beta_ind_Government,beta_ind_Industrial,beta_ind_Technology,beta_ind_Utilities
0,000307108,1251,-0.001609,0.975736,0.126150,0.057386,-8.437695e-14,-0.052679,-0.084945,-0.940445,...,-0.327868,0.277601,-0.022685,0.068375,-0.771996,-2.671759,0.110655,0.221339,-0.847782,-0.350057
1,00032Q104,1338,-0.000463,0.793721,0.007701,0.295796,-2.436940e-14,0.044163,0.191688,0.571437,...,0.041280,0.891037,-0.020420,0.345415,0.633918,-1.992948,0.413623,-0.801888,-0.620049,-0.226723
2,000360206,5468,0.000529,1.118369,-0.010468,0.045201,-5.988268e-03,0.001915,-0.000182,-0.198529,...,0.091248,0.024453,-0.005766,-0.082550,-0.066306,-0.027480,-0.001746,0.613039,-0.127176,-0.094830
3,000361105,5481,0.000223,1.425338,0.057237,-0.009537,6.099019e-02,0.098455,0.017490,0.036780,...,0.465824,0.305997,0.040661,0.427869,0.435034,0.570607,-0.006007,1.320757,0.328607,0.075798
4,000375204,5272,0.000165,1.241287,0.003964,0.026373,4.576894e-02,0.061782,-0.002554,0.192800,...,0.042191,-0.201559,0.021117,-0.112964,0.147023,-0.037118,-0.000816,0.363655,0.050011,0.020682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19954,Y92335101,363,0.006010,4.669363,-0.368823,-0.717647,2.972622e-13,-1.120035,-0.290594,-5.156823,...,-2.759075,2.109515,1.149566,-0.450108,1.675461,-20.168260,-0.061767,0.826269,-2.475183,-1.477284
19955,Y93691106,1245,0.000474,0.971725,-0.089492,-0.115024,-3.848916e-02,-0.161895,-0.201610,-0.171775,...,-0.117512,-0.527249,0.039492,-0.389095,-0.137936,0.229872,-0.001223,-0.072022,0.437670,-0.084952
19956,Y9384M101,706,0.000684,0.650243,0.061451,0.202190,1.444678e-14,0.141448,0.193190,0.485212,...,0.192799,0.670232,0.001182,0.985603,0.581781,-3.317787,-0.058742,0.297592,0.267774,0.342143
19957,Y9390M103,93,0.007187,3.639796,1.019682,1.761116,-4.756195e-13,-5.268887,3.403726,7.796101,...,-11.209055,-20.612412,-10.142711,-5.131451,-2.553662,-54.510123,3.306581,-11.114412,-29.249670,-3.528161


In [23]:
loadings_long_df

,cusip9,factor,beta
0,000307108,market_factor_return,9.757359e-01
1,000307108,cty_AR,1.261497e-01
2,000307108,cty_AU,5.738611e-02
3,000307108,cty_BD,-8.437695e-14
4,000307108,cty_BE,-5.267911e-02
...,...,...,...
1337248,Y95308105,ind_Funds,-3.784748e+00
1337249,Y95308105,ind_Government,-2.763163e-01
1337250,Y95308105,ind_Industrial,1.728149e-01
1337251,Y95308105,ind_Technology,-2.665429e-01


In [25]:
from pathlib import Path

# Export wide and long loadings to separate CSV files
# Reuse existing out_dir if available, otherwise create ./outputs
if 'loadings_wide_df' not in globals() or 'loadings_long_df' not in globals():
    raise RuntimeError("loadings_wide_df and/or loadings_long_df not found in the notebook namespace")

if 'out_dir' in globals():
    target_dir = out_dir
else:
    target_dir = Path.cwd() / 'outputs'
target_dir.mkdir(parents=True, exist_ok=True)

# prefer an existing "Output"/"outputs" folder near the notebook; search cwd and parents

def find_existing_output(start: Path):
    for p in [start] + list(start.parents):
        for name in ('Output','output','outputs','Outputs'):
            cand = p / name
            if cand.is_dir():
                return cand
    return None

existing = find_existing_output(Path.cwd())
if existing is not None:
    wide_path = existing / 'cusip_loadings_wide.csv'
    target_dir = existing
else:
    wide_path = target_dir / 'cusip_loadings_wide.csv'
long_path = target_dir / 'cusip_loadings_long.csv'

loadings_wide_df.to_csv(wide_path, index=False)
loadings_long_df.to_csv(long_path, index=False)

print("Saved wide loadings to:", wide_path, "rows:", len(loadings_wide_df))
print("Saved long loadings to:", long_path, "rows:", len(loadings_long_df))

Saved wide loadings to: /Users/teddrako/Documents/GitHub/multi-factor-models/ted_workspace/Research/outputs/cusip_loadings_wide.csv rows: 19959
Saved long loadings to: /Users/teddrako/Documents/GitHub/multi-factor-models/ted_workspace/Research/outputs/cusip_loadings_long.csv rows: 1337253
